# Cold-start short-video recommendation: end to end

Runs the whole pipeline on a free Colab T4: download, tests, embeddings, baselines, all models x 3 seeds, the tables and figures, then the vector-length test.

**Runtime > Change runtime type > T4 GPU** before running. The full run takes several hours; results are written as each step finishes, so a session that is cut off still leaves the finished runs in `results/`.

In [ ]:
REPO_URL = "https://github.com/FireCloud24/shortvideo-coldstart.git"  # change if your fork lives elsewhere
SEEDS = "0 1 2"
EXTRAS = 0  # 1 also runs English CLIP titles, LoRA and Qwen2-VL (slow; not needed for the finding)
IMAGE_LORA_STEPS = 3000  # step budget for LoRA on the image tower (the slow stage)

import os
# Absolute paths, so re-running this cell does not clone a second copy inside the first.
if not os.path.exists('/content/shortvideo-coldstart'):
    !git clone -q $REPO_URL /content/shortvideo-coldstart
%cd /content/shortvideo-coldstart
!git log -1 --format='code version: %h %s'
!pip install -q -r requirements.txt
# Colab ships an old torchao that makes peft refuse to import; this project does not use torchao.
!pip uninstall -y -q torchao
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

## 0. Keep the results safe (recommended)
A free Colab session can end without warning, and everything on its disk is then lost. With `USE_DRIVE = True` the run copies results, feature files and checkpoints to a folder on your Google Drive after every step, and a later session picks up where the last one stopped. Colab will ask you to allow access to Drive. Delete the Drive folder to start from scratch (do this after any change to the code).

In [ ]:
USE_DRIVE = True
PERSIST_DIR = ''
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST_DIR = '/content/drive/MyDrive/shortvideo-coldstart-run2'
print('saving to:', PERSIST_DIR or 'this session only')

## 1. Tests

In [ ]:
!python -m pytest -q

## 2. Download MicroLens-100K
About 710 MB. Re-running skips files already present.

In [ ]:
!python scripts/download.py
!cat data/raw/manifest.json | head -20

## 3. Smoke test
A few items through every encoder and one training epoch, so a problem shows up in minutes rather than hours.

In [ ]:
!python scripts/embed.py --model cnclip --limit 64
!python scripts/embed.py --model enclip --limit 64
!python scripts/train.py --config configs/sasrec_id.yaml --seeds 0 --set train.epochs=1 out_dir=results_smoke

## 4. Every experiment
`scripts/run_all.sh` runs the baselines, the content models, LoRA and the report, cheapest first. The full log goes to `run_all.log`; the next cell prints its summary lines.

In [ ]:
import os
# The repository's results/ folder holds an earlier run; remove it so no old number survives.
!rm -rf results/runs results/finetune results/features results/lr_search results/*.md results/*.png
if PERSIST_DIR and os.path.isdir(f'{PERSIST_DIR}/results'):
    print('resuming from', PERSIST_DIR)
    !mkdir -p results data/features checkpoints
    !cp -r "{PERSIST_DIR}/results/." results/
    !cp -r "{PERSIST_DIR}/features/." data/features/
    !cp -r "{PERSIST_DIR}/checkpoints/." checkpoints/
    !cp "{PERSIST_DIR}/run_all.log" run_all.log
    !echo "=== resumed in a new session ===" >> run_all.log
!SEEDS="{SEEDS}" EXTRAS={EXTRAS} IMAGE_LORA_STEPS={IMAGE_LORA_STEPS} PERSIST_DIR="{PERSIST_DIR}" bash scripts/run_all.sh >> run_all.log 2>&1
!cp run_all.log results/run_all.log
!grep -E '^=== |seed [0-9]+: |^!!! |^wrote |cost: ' run_all.log

## 5. Report

In [ ]:
from IPython.display import Markdown, Image, display
for name in ('tradeoff', 'exposure', 'categories'):
    display(Image(f'results/{name}.png'))
    display(Markdown(open(f'results/{name}.md').read()))
display(Markdown(open('results/results.md').read()))
display(Image('results/warm_vs_cold.png'))
display(Markdown(open('results/error_slices.md').read()))
display(Image('results/error_slices.png'))

## 5b. Vector-length test
Retrains two ID + content models with their weights saved and multiplies the new videos' vectors by a factor, with nothing retrained. About half an hour.

In [ ]:
!bash scripts/run_scale_test.sh > scale_test.log 2>&1
!grep -E '^=== |as trained|^!!! |^wrote ' scale_test.log
from IPython.display import Markdown, Image, display
display(Image('results/scale_test/scale_test.png'))
display(Markdown(open('results/scale_test/scale_test.md').read()))

## 6. Save the results
`results/` holds only metrics, per-example ranks, tables, charts and the run log (no dataset files), so it can be committed. Download the zip and commit it from your machine, together with this notebook (File > Download > .ipynb) so the outputs above are kept.

In [ ]:
!zip -q -r results.zip results
from google.colab import files
files.download('results.zip')